# Phase 1 — OpenML Data Pull

Pulls 60–70 tabular classification datasets from OpenML, caches them locally, and
saves a manifest CSV listing each dataset's key properties.

**Filters** (from CLAUDE.md):
- 100 ≤ n_instances ≤ 100,000
- 2 ≤ n_classes ≤ 10
- n_features < 200
- No missing values
- Tabular / classification tasks only

**Output**: `data/meta_table/dataset_manifest.csv`

**Showcase datasets are excluded** from the manifest to prevent training leakage.

In [ ]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import openml

# ── Paths ────────────────────────────────────────────────────────────────────
def find_project_root(start=None):
    cur = Path(start or os.getcwd()).resolve()
    for path in (cur, *cur.parents):
        if (path / 'CLAUDE.md').exists() and (path / 'src').exists():
            return str(path)
    raise RuntimeError('Project root not found; run from the repo root or notebooks folder')

ROOT        = find_project_root()
RAW_DIR     = os.path.join(ROOT, 'data', 'raw')
META_DIR    = os.path.join(ROOT, 'data', 'meta_table')
MANIFEST    = os.path.join(META_DIR, 'dataset_manifest.csv')

os.makedirs(RAW_DIR,  exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

# Cache OpenML downloads to data/raw/
openml.config.cache_directory = RAW_DIR

print(f'ROOT     : {ROOT}')
print(f'RAW_DIR  : {RAW_DIR}')
print(f'MANIFEST : {MANIFEST}')

In [3]:
# ── Showcase dataset IDs — NEVER include these in meta-training ───────────────
# Identified by OpenML dataset ID (authoritative — names can alias)
SHOWCASE_IDS = {
    61,    # iris
    187,   # wine
    15,    # breast-w (Breast Cancer Wisconsin)
    53,    # heart-statlog (Heart Disease UCI)
    40966, # penguins (Palmer Penguins)
    37,    # diabetes (Pima)
    54,    # vehicle (Vehicle Silhouettes)
    1590,  # adult (Adult Income)
    1597,  # creditcard (Credit Card Fraud)
}

print(f'Showcase IDs excluded from training: {sorted(SHOWCASE_IDS)}')

Showcase IDs excluded from training: [15, 37, 53, 54, 61, 187, 1590, 1597, 40966]


In [4]:
# ── Filter constants (from CLAUDE.md) ────────────────────────────────────────
MIN_INSTANCES  = 100
MAX_INSTANCES  = 100_000
MIN_CLASSES    = 2
MAX_CLASSES    = 10
MAX_FEATURES   = 200
TARGET_COUNT   = 70   # aim for 60–70; stop early if reached

print('Filters:')
print(f'  instances : {MIN_INSTANCES:,} – {MAX_INSTANCES:,}')
print(f'  classes   : {MIN_CLASSES} – {MAX_CLASSES}')
print(f'  features  : < {MAX_FEATURES}')
print(f'  missing   : none allowed')

Filters:
  instances : 100 – 100,000
  classes   : 2 – 10
  features  : < 200
  missing   : none allowed


In [5]:
# ── Fetch OpenML task list (supervised classification) ────────────────────────
# Task type 1 = Supervised Classification on OpenML
print('Fetching task list from OpenML (this may take ~30 s on first run)...')

tasks = openml.tasks.list_tasks(
    task_type=openml.tasks.TaskType.SUPERVISED_CLASSIFICATION,
    output_format='dataframe',
)

print(f'Total tasks returned: {len(tasks):,}')
tasks.head(3)

Fetching task list from OpenML (this may take ~30 s on first run)...
Total tasks returned: 5,581


,tid,ttid,did,name,task_type,status,estimation_procedure,evaluation_measures,source_data,target_feature,...,MaxNominalAttDistinctValues,MinorityClassSize,NumberOfClasses,NumberOfFeatures,NumberOfInstances,NumberOfInstancesWithMissingValues,NumberOfMissingValues,NumberOfNumericFeatures,NumberOfSymbolicFeatures,cost_matrix
1,1,TaskType.SUPERVISED_CLASSIFICATION,1,anneal,Supervised Classification,active,10-fold Crossvalidation,predictive_accuracy,1,class,...,8.0,8.0,5.0,39.0,898.0,0.0,0.0,6.0,33.0,NaN
2,2,TaskType.SUPERVISED_CLASSIFICATION,2,anneal,Supervised Classification,active,10-fold Crossvalidation,predictive_accuracy,2,class,...,7.0,8.0,5.0,39.0,898.0,898.0,22175.0,6.0,33.0,NaN
3,3,TaskType.SUPERVISED_CLASSIFICATION,3,kr-vs-kp,Supervised Classification,active,10-fold Crossvalidation,NaN,3,class,...,3.0,1527.0,2.0,37.0,3196.0,0.0,0.0,0.0,37.0,NaN


In [6]:
# ── Inspect available columns for filtering ───────────────────────────────────
print('Columns:', tasks.columns.tolist())
print()
# Show which size/quality columns are present
for col in ['NumberOfInstances', 'NumberOfFeatures', 'NumberOfClasses',
            'NumberOfMissingValues', 'did', 'name']:
    present = col in tasks.columns
    print(f'  {col:35s} present={present}')

Columns: ['tid', 'ttid', 'did', 'name', 'task_type', 'status', 'estimation_procedure', 'evaluation_measures', 'source_data', 'target_feature', 'MajorityClassSize', 'MaxNominalAttDistinctValues', 'MinorityClassSize', 'NumberOfClasses', 'NumberOfFeatures', 'NumberOfInstances', 'NumberOfInstancesWithMissingValues', 'NumberOfMissingValues', 'NumberOfNumericFeatures', 'NumberOfSymbolicFeatures', 'cost_matrix']

  NumberOfInstances                   present=True
  NumberOfFeatures                    present=True
  NumberOfClasses                     present=True
  NumberOfMissingValues               present=True
  did                                 present=True
  name                                present=True


In [7]:
# ── Apply size / quality filters ──────────────────────────────────────────────
df = tasks.copy()

# Drop rows with missing filter columns
needed = ['NumberOfInstances', 'NumberOfFeatures', 'NumberOfClasses',
          'NumberOfMissingValues', 'did']
df = df.dropna(subset=needed)

# Cast to numeric
for col in needed[:-1]:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=needed)

before = len(df)
df = df[
    (df['NumberOfInstances']     >= MIN_INSTANCES) &
    (df['NumberOfInstances']     <= MAX_INSTANCES) &
    (df['NumberOfClasses']       >= MIN_CLASSES)   &
    (df['NumberOfClasses']       <= MAX_CLASSES)   &
    (df['NumberOfFeatures']      <  MAX_FEATURES)  &
    (df['NumberOfMissingValues'] == 0)
]
print(f'After size/quality filters: {len(df):,}  (from {before:,})')

After size/quality filters: 2,748  (from 5,520)


In [8]:
# ── Deduplicate by dataset ID, exclude showcase IDs ──────────────────────────
df = df.drop_duplicates(subset='did')
df = df[~df['did'].isin(SHOWCASE_IDS)]

# Sort by instance count for diversity; reset index
df = df.sort_values('NumberOfInstances').reset_index(drop=True)

print(f'After dedup + showcase exclusion: {len(df):,} candidate datasets')

After dedup + showcase exclusion: 1,312 candidate datasets


In [9]:
# ── Select a diverse subset of up to TARGET_COUNT datasets ───────────────────
# Strategy: bin by log-instance-count and sample uniformly across size bins
# so the meta-training table covers small, medium, and large datasets.

import math

df['log_n'] = np.log10(df['NumberOfInstances'])

# 5 bins across the log-instance range
n_bins = 5
df['size_bin'] = pd.cut(df['log_n'], bins=n_bins, labels=False)

per_bin = math.ceil(TARGET_COUNT / n_bins)
sampled = (
    df.groupby('size_bin', group_keys=False)
      .apply(lambda g: g.sample(min(per_bin, len(g)), random_state=42))
)

# Trim to exactly TARGET_COUNT if over
sampled = sampled.sample(frac=1, random_state=42).head(TARGET_COUNT).reset_index(drop=True)

print(f'Selected {len(sampled)} datasets for meta-training')
print(sampled[['did', 'name', 'NumberOfInstances', 'NumberOfFeatures', 'NumberOfClasses']].to_string(index=False))

Selected 70 datasets for meta-training
  did                                                                                    name  NumberOfInstances  NumberOfFeatures  NumberOfClasses
43924                                                                              eucalyptus              736.0              20.0              5.0
  732                                                                           fri_c0_250_50              250.0              51.0              2.0
 1459                                                                   artificial-characters            10218.0               8.0             10.0
 1495                                                                  qualitative-bankruptcy              250.0               7.0              2.0
 4538                                                       GesturePhaseSegmentationProcessed             9873.0              33.0              5.0
46176                                                                    

In [10]:
# ── Download & verify each dataset ───────────────────────────────────────────
# Downloads are cached by the OpenML library under RAW_DIR.
# We verify that each dataset loads correctly and record actual row/col counts.

records = []
failed  = []

for i, row in sampled.iterrows():
    did  = int(row['did'])
    name = row.get('name', '')
    try:
        ds     = openml.datasets.get_dataset(
                     did,
                     download_data=True,
                     download_qualities=True,
                     download_features_meta_data=False,
                 )
        X, y, _, _  = ds.get_data(dataset_format='dataframe',
                                   target=ds.default_target_attribute)

        # Verify no missing values after load
        if X.isnull().any().any() or (y is not None and y.isnull().any()):
            failed.append((did, name, 'missing values after load'))
            continue

        n_inst    = X.shape[0]
        n_feat    = X.shape[1]
        n_classes = int(y.nunique()) if y is not None else 0

        records.append({
            'dataset_id' : did,
            'name'       : ds.name,
            'n_instances': n_inst,
            'n_features' : n_feat,
            'n_classes'  : n_classes,
        })

        print(f'[{i+1:3d}/{len(sampled)}] OK  id={did:6d}  {ds.name[:40]:40s}  '
              f'{n_inst:6d}r × {n_feat:3d}c  {n_classes}cls')

    except Exception as e:
        failed.append((did, name, str(e)))
        print(f'[{i+1:3d}/{len(sampled)}] FAIL id={did}  {name}  — {e}')

print(f'\nLoaded: {len(records)}  Failed: {len(failed)}')

[  1/70] OK  id= 43924  eucalyptus                                   736r ×  19c  5cls
[  2/70] OK  id=   732  fri_c0_250_50                                250r ×  50c  2cls
[  3/70] OK  id=  1459  artificial-characters                      10218r ×   7c  10cls
[  4/70] OK  id=  1495  qualitative-bankruptcy                       250r ×   6c  2cls
[  5/70] OK  id=  4538  GesturePhaseSegmentationProcessed           9873r ×  32c  5cls
[  6/70] OK  id= 46176  Flare                                       1066r ×  11c  6cls
[  7/70] OK  id=   694  diggle_table_a2                              310r ×   8c  9cls
[  8/70] OK  id= 43958  kdd_ipums_la_97-small                       5188r ×  20c  2cls
[  9/70] OK  id=  1019  pendigits                                  10992r ×  16c  2cls
[ 10/70] OK  id= 42544  Touch2                                       265r ×  10c  8cls
[ 11/70] OK  id= 46944  Mobile_Price                                2000r ×  20c  4cls
[ 12/70] OK  id=  1446  CostaMadre1       

In [ ]:
# ── Save manifest ─────────────────────────────────────────────────────────────
manifest = pd.DataFrame(records)
manifest.to_csv(MANIFEST, index=False)

print(f'Manifest saved → {MANIFEST}')
print(f'Shape: {manifest.shape}')
manifest

In [12]:
# ── Sanity checks ─────────────────────────────────────────────────────────────

# 1. No showcase IDs must appear in the manifest
leaked = set(manifest['dataset_id']) & SHOWCASE_IDS
assert len(leaked) == 0, f'SHOWCASE LEAK: {leaked}'
print('✓ No showcase IDs in manifest')

# 2. All instances within allowed range
assert manifest['n_instances'].between(MIN_INSTANCES, MAX_INSTANCES).all(), \
    'Instance count out of range'
print('✓ All datasets within instance range')

# 3. All class counts within allowed range
assert manifest['n_classes'].between(MIN_CLASSES, MAX_CLASSES).all(), \
    'Class count out of range'
print('✓ All datasets within class range')

# 4. All feature counts below max
assert (manifest['n_features'] < MAX_FEATURES).all(), \
    'Feature count out of range'
print('✓ All datasets below feature limit')

# 5. No duplicate IDs
assert manifest['dataset_id'].is_unique, 'Duplicate dataset IDs'
print('✓ No duplicate dataset IDs')

# 6. Count
print(f'\nFinal manifest: {len(manifest)} datasets ready for Phase 2 (LSE computation)')

✓ No showcase IDs in manifest
✓ All datasets within instance range
✓ All datasets within class range
✓ All datasets below feature limit
✓ No duplicate dataset IDs

Final manifest: 69 datasets ready for Phase 2 (LSE computation)


In [13]:
# ── Summary statistics ────────────────────────────────────────────────────────
print('=== Dataset manifest summary ===')
print(manifest[['n_instances','n_features','n_classes']].describe().round(1).to_string())

print('\n=== Class distribution ===')
print(manifest['n_classes'].value_counts().sort_index())

if failed:
    print('\n=== Failed downloads ===')
    for did, name, reason in failed:
        print(f'  id={did}  {name}  — {reason}')

=== Dataset manifest summary ===
       n_instances  n_features  n_classes
count         69.0        69.0       69.0
mean       13049.8        21.7        3.2
std        20756.4        19.6        2.1
min          100.0         1.0        2.0
25%          666.0         8.0        2.0
50%         2000.0        11.0        2.0
75%        13750.0        32.0        4.0
max        83733.0       100.0       10.0

=== Class distribution ===
n_classes
2     46
3      3
4      4
5      8
6      2
7      1
8      2
9      1
10     2
Name: count, dtype: int64

=== Failed downloads ===
  id=350  webdata_wXa  — factorize requires a Series, Index, ExtensionArray, np.ndarray or NumpyExtensionArray got list.


In [15]:
# ── Replace failed dataset 350 (webdata_wXa — malformed target column) ────────
# Candidates tried in order; first successful download wins.
REPLACEMENT_CANDIDATES = [
    32,   # satimage   — 6430r × 36f, 6cls
    24,   # mushroom   — 8124r × 22f, 2cls
    29,   # page-blocks — 5473r × 10f, 5cls
    28,   # optdigits  — 5620r × 64f, 10cls
]

import pandas as pd
import os

MANIFEST = os.path.join(ROOT, 'data', 'meta_table', 'dataset_manifest.csv')
SHOWCASE_IDS = {
    61, 187, 15, 53, 40966, 37, 54, 1590, 1597,
}

manifest = pd.read_csv(MANIFEST)
existing_ids = set(manifest['dataset_id'])

replacement = None
for cand_id in REPLACEMENT_CANDIDATES:
    if cand_id in existing_ids or cand_id in SHOWCASE_IDS:
        print(f'  Skip {cand_id} — already in manifest or showcase')
        continue
    try:
        import openml
        ds = openml.datasets.get_dataset(
                 cand_id,
                 download_data=True,
                 download_qualities=True,
                 download_features_meta_data=False,
             )
        X, y, _, _ = ds.get_data(dataset_format='dataframe',
                                  target=ds.default_target_attribute)
        if X.isnull().any().any() or (y is not None and y.isnull().any()):
            print(f'  Skip {cand_id} ({ds.name}) — missing values after load')
            continue
        n_inst    = X.shape[0]
        n_feat    = X.shape[1]
        n_classes = int(y.nunique()) if y is not None else 0
        # Verify filters
        if not (100 <= n_inst <= 100_000 and 2 <= n_classes <= 10 and n_feat < 200):
            print(f'  Skip {cand_id} ({ds.name}) — fails filter: '
                  f'{n_inst}r {n_feat}f {n_classes}cls')
            continue
        replacement = {
            'dataset_id' : cand_id,
            'name'       : ds.name,
            'n_instances': n_inst,
            'n_features' : n_feat,
            'n_classes'  : n_classes,
        }
        print(f'Replacement: id={cand_id}  {ds.name}  {n_inst}r × {n_feat}f  {n_classes}cls')
        break
    except Exception as e:
        print(f'  Skip {cand_id} — {e}')

assert replacement is not None, 'All replacement candidates failed — add more to the list'

manifest = pd.concat([manifest, pd.DataFrame([replacement])], ignore_index=True)
manifest.to_csv(MANIFEST, index=False)

print(f'Manifest updated: {len(manifest)} datasets total')
assert replacement['dataset_id'] not in SHOWCASE_IDS, 'Showcase leak!'
assert manifest['dataset_id'].is_unique, 'Duplicate IDs!'
print('Sanity checks passed.')


Replacement: id=32  pendigits  10992r × 16f  10cls
Manifest updated: 70 datasets total
Sanity checks passed.
